In [ ]:
import pandas as pd
import requests
import re
import time
from pathlib import Path
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')

## Configuration

In [ ]:
# Paths
CSV_PATH = Path('../literature/data/scraped.csv')
PDF_DIR = Path('../literature/pdfs')
PDF_DIR.mkdir(parents=True, exist_ok=True)

# Download settings
DELAY_BETWEEN_REQUESTS = 1.5  # seconds
REQUEST_TIMEOUT = 30  # seconds
# Don't fake user agent...
# USER_AGENT = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'

## Load Data

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} entries")
df.head()

## Generate Google Scholar-style IDs

In [ ]:
def generate_id(row):
    """
    Generate Google Scholar-style ID: lastname + year + first_word
    Example: 'barreto2025ex'
    """
    # Extract first author's last name
    authors = str(row['authors'])
    if pd.isna(authors) or authors == 'nan':
        lastname = 'unknown'
    else:
        # Split by semicolon or comma and take first author
        first_author = authors.split(';')[0].split(',')[0].strip()
        # Take last name (after last space or the whole thing)
        lastname = first_author.split()[-1] if ' ' in first_author else first_author
        lastname = lastname.lower()
        # Remove special characters
        lastname = re.sub(r'[^a-z]', '', lastname)

    # Extract year
    year = str(row['year']) if not pd.isna(row['year']) else 'unknown'

    # Extract first word from title (excluding common articles)
    title = str(row['title'])
    if pd.isna(title) or title == 'nan':
        first_word = 'unknown'
    else:
        # Remove quotes and split
        title_clean = re.sub(r'["\']', '', title)
        words = title_clean.split()
        # Skip common articles/prepositions
        skip_words = {'the', 'a', 'an', 'of', 'in', 'on', 'at', 'to', 'for'}
        first_word = next((w for w in words if w.lower() not in skip_words), words[0] if words else 'unknown')
        first_word = first_word.lower()
        # Remove special characters
        first_word = re.sub(r'[^a-z]', '', first_word)

    return f"{lastname}{year}{first_word}"

df['id'] = df.apply(generate_id, axis=1)

# Check for duplicates and make unique
duplicates = df['id'].duplicated(keep=False)
if duplicates.any():
    print(f"Found {duplicates.sum()} duplicate IDs, making them unique...")
    for idx in df[duplicates].index:
        counter = 1
        base_id = df.loc[idx, 'id']
        while df.loc[idx, 'id'] in df.loc[:idx-1, 'id'].values:
            df.loc[idx, 'id'] = f"{base_id}_{counter}"
            counter += 1

print(f"\nGenerated IDs for {len(df)} entries")
print(f"Sample IDs: {df['id'].head(5).tolist()}")

## PDF Download Functions

In [ ]:
def get_session():
    """Create a requests session with headers"""
    session = requests.Session()
    session.headers.update({
        # 'User-Agent': USER_AGENT,
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,application/pdf,*/*;q=0.8',
    })
    return session

def is_pdf_url(url):
    """Check if URL likely points to a PDF"""
    if pd.isna(url) or url == 'nan':
        return False
    return url.lower().endswith('.pdf') or '.pdf' in url.lower()

def download_pdf_direct(url, output_path, session):
    """Try to download PDF directly from URL"""
    try:
        response = session.get(url, timeout=REQUEST_TIMEOUT, allow_redirects=True)

        # Check if response is PDF
        content_type = response.headers.get('Content-Type', '').lower()
        if 'application/pdf' in content_type or response.content[:4] == b'%PDF':
            with open(output_path, 'wb') as f:
                f.write(response.content)
            return True
    except Exception as e:
        print(f"  Error downloading directly: {e}")
    return False

def find_pdf_links_on_page(url, session):
    """Parse HTML page to find PDF download links (e.g., 'Download PDF' buttons)"""
    try:
        response = session.get(url, timeout=REQUEST_TIMEOUT)
        soup = BeautifulSoup(response.text, 'html.parser')

        pdf_candidates = []

        # Method 1: Look for direct PDF links
        for link in soup.find_all('a', href=True):
            href = link['href']
            if '.pdf' in href.lower():
                absolute_url = urljoin(url, href)
                pdf_candidates.append(absolute_url)

        # Method 2: Look for common PDF download button/link text
        pdf_keywords = [
            'download pdf', 'pdf download', 'full text pdf', 'download article',
            'view pdf', 'get pdf', 'pdf version', 'download full text',
            'volltext', 'herunterladen'
        ]
        for link in soup.find_all('a', href=True):
            link_text = link.get_text().strip().lower()
            # Also check aria-label and title attributes
            aria_label = link.get('aria-label', '').lower()
            title = link.get('title', '').lower()
            combined_text = f"{link_text} {aria_label} {title}"

            if any(keyword in combined_text for keyword in pdf_keywords):
                absolute_url = urljoin(url, link['href'])
                if absolute_url not in pdf_candidates:
                    # Prioritize links that seem like direct downloads
                    if '.pdf' in absolute_url.lower():
                        pdf_candidates.insert(0, absolute_url)
                    else:
                        pdf_candidates.append(absolute_url)

        # Method 3: Look for meta tags pointing to PDFs
        for meta in soup.find_all('meta'):
            content = meta.get('content', '')
            if '.pdf' in content.lower():
                absolute_url = urljoin(url, content)
                if absolute_url not in pdf_candidates:
                    pdf_candidates.append(absolute_url)

        return pdf_candidates
    except Exception as e:
        print(f"  Error parsing HTML: {e}")
        return []

def try_doi_pdf(doi, session):
    """Try to get PDF from DOI via direct publisher access"""
    if pd.isna(doi) or doi == 'nan':
        return None

    # Extract DOI if it's a full URL
    doi_match = re.search(r'10\.\d{4,}/[^\s]+', doi)
    if doi_match:
        clean_doi = doi_match.group(0)

        # Try publisher direct link (often works for open access)
        publisher_urls = [
            f"https://doi.org/{clean_doi}",
        ]

        return publisher_urls
    return None


## Main Download Logic

In [ ]:
def download_paper_pdf(row, session):
    """
    Try multiple strategies to download PDF for a paper
    Returns: (success: bool, message: str)
    """
    paper_id = row['id']
    output_path = PDF_DIR / f"{paper_id}.pdf"

    # Skip if already downloaded
    if output_path.exists():
        return True, "Already exists"

    print(f"\nProcessing: {paper_id}")
    print(f"  Title: {row['title'][:60]}...")

    # Strategy 1: Check if doi_link points directly to PDF
    doi_link = row.get('doi_link', '')
    if is_pdf_url(doi_link):
        print("  Strategy 1: Direct PDF from DOI link")
        if download_pdf_direct(doi_link, output_path, session):
            return True, "Downloaded from DOI link"

    # Strategy 2: If doi_link is not a PDF, parse the page for PDF download links
    if doi_link and not pd.isna(doi_link) and doi_link != 'nan':
        print("  Strategy 2: Looking for PDF links on DOI page")
        pdf_links = find_pdf_links_on_page(doi_link, session)
        for pdf_link in pdf_links[:5]:  # Try first 5 candidates
            if download_pdf_direct(pdf_link, output_path, session):
                return True, "Downloaded from PDF link on page"
            time.sleep(0.5)

    # Strategy 3: Try DOI resolution (if DOI is provided)
    doi = row.get('doi_link', '')
    doi_urls = try_doi_pdf(doi, session)
    if doi_urls:
        print("  Strategy 3: Trying DOI resolution")
        for doi_url in doi_urls:
            if download_pdf_direct(doi_url, output_path, session):
                return True, "Downloaded via DOI resolution"
            # Try to find PDF link on the resolved DOI page
            pdf_links = find_pdf_links_on_page(doi_url, session)
            for pdf_link in pdf_links[:5]:  # Try first 5 candidates
                if download_pdf_direct(pdf_link, output_path, session):
                    return True, "Downloaded from resolved DOI page link"
                time.sleep(0.5)

    return False, "No PDF found"

# Download all PDFs
session = get_session()
results = []

for idx, row in df.iterrows():
    success, message = download_paper_pdf(row, session)
    results.append({
        'id': row['id'],
        'title': row['title'],
        'success': success,
        'message': message
    })
    print(f"  Result: {message}")

    # Respectful delay between requests
    if idx < len(df) - 1:
        time.sleep(DELAY_BETWEEN_REQUESTS)


## Summary

In [ ]:
results_df = pd.DataFrame(results)

total = len(results_df)
successful = results_df['success'].sum()
already_existed = (results_df['message'] == 'Already exists').sum()
newly_downloaded = successful - already_existed
failed = total - successful

print(f"\n{'='*60}")
print("DOWNLOAD SUMMARY")
print(f"{'='*60}")
print(f"Total papers: {total}")
print(f"Successfully available: {successful} ({successful/total*100:.1f}%)")
print(f"  - Already existed: {already_existed}")
print(f"  - Newly downloaded: {newly_downloaded}")
print(f"Failed to download: {failed} ({failed/total*100:.1f}%)")
print(f"\nPDFs saved to: {PDF_DIR.absolute()}")

# Show failed downloads
if failed > 0:
    print("\nFailed downloads:")
    failed_df = results_df[~results_df['success']]
    for _, row in failed_df.iterrows():
        print(f"  - {row['id']}: {row['title'][:50]}...")

## Save Results

In [ ]:
# Save the dataframe with IDs for future reference
output_csv = Path('../literature/data/scraped_with_ids.csv')
df.to_csv(output_csv, index=False)
print(f"Saved data with IDs to: {output_csv}")

# Save download results
results_csv = Path('../literature/data/download_results.csv')
results_df.to_csv(results_csv, index=False)
print(f"Saved download results to: {results_csv}")

## Manual


In [ ]:
# Filter for rows where PDF doesn't exist
data = pd.read_csv('../literature/data/scraped_with_ids.csv')
missing_pdfs = data[~data['id'].apply(lambda x: (PDF_DIR / f"{x}.pdf").exists())]

print(f"Found {len(missing_pdfs)} papers without PDFs\n")

In [ ]:
import subprocess
import webbrowser

if len(missing_pdfs) == 0:
    print("All PDFs are already downloaded!")
else:
    for idx, row in missing_pdfs.iterrows():
        paper_id = row['id']
        title = row['title']
        doi_link = row.get('doi_link', '')

        print(f"\n{'='*70}")
        print(f"ID: {paper_id}")
        print(f"Title: {title}")
        print(f"Link: {doi_link if doi_link and doi_link != 'nan' else 'No link available'}")
        print(f"{'='*70}\n")

        # Copy ID to clipboard (macOS)
        subprocess.run(['pbcopy'], input=paper_id.encode(), check=True)
        print(f"✓ ID '{paper_id}' copied to clipboard")

        # Open link in browser if available
        if doi_link and doi_link != 'nan' and not pd.isna(doi_link):
            webbrowser.open(doi_link)
            print("✓ Opened link in browser")
        else:
            print("✗ No link available to open")

        # Wait for user input
        input("\nPress Enter to continue to next paper...")

    print(f"\n\nCompleted reviewing {len(missing_pdfs)} papers with missing PDFs")